In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
import os
import re
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms

# ---------------------------------------------------------
# 1. Mapping-ul pentru etichetele din JAFFE (7 expresii)
# ---------------------------------------------------------
EMOTION_MAP = {
    'AN': 0,  # Anger
    'DI': 1,  # Disgust
    'FE': 2,  # Fear
    'HA': 3,  # Happy
    'NE': 4,  # Neutral
    'SA': 5,  # Sad
    'SU': 6   # Surprise
}

# ---------------------------------------------------------
# 2. Definitia PyTorch Dataset pentru JAFFE
# ---------------------------------------------------------
class JAFFEDataset(Dataset):
    """
    Dataset custom pentru incarcarea imaginilor JAFFE.
    Extrage eticheta direct din numele fisierelor (ex: KA.AN1.39.tiff).
    """
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.images_paths = []
        self.labels = []

        # Extragere fisiere si etichete
        valid_extensions = ('.tiff', '.tif', '.png', '.jpg', '.jpeg')
        for file_name in os.listdir(root_dir):
            if file_name.lower().endswith(valid_extensions):
                # Regex pentru a extrage codul de 2 litere al emotiei (ex: AN, HA)
                match = re.search(r'\.([A-Z]{2})\d', file_name)
                if match:
                    emotion_code = match.group(1)
                    if emotion_code in EMOTION_MAP:
                        self.images_paths.append(os.path.join(root_dir, file_name))
                        self.labels.append(EMOTION_MAP[emotion_code])

    def __len__(self):
        return len(self.images_paths)

    def __getitem__(self, idx):
        img_path = self.images_paths[idx]
        # Convertim la RGB intrucat ResNet asteapta 3 canale
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

# ---------------------------------------------------------
# 3. Transformari / Preprocesare si Augmentari
# ---------------------------------------------------------
# Transforma standard pentru Validare si Test (fara augmentare aleatoare)
val_test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Transforma pentru Antrenare (include augmentari clasice pentru a preveni overfitting-ul)
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# ---------------------------------------------------------
# 4. Functie helper pentru crearea Split-urilor si DataLoader-elor
# ---------------------------------------------------------
def get_jaffe_dataloaders(data_dir, batch_size=16, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15, seed=42):
    """
    Incarca setul de date, il imparte in Train / Val / Test si returneaza DataLoader-ele aferente.
    """
    torch.manual_seed(seed)

    # Dataset complet initial (fara transformari aplicate direct pe tot obiectul pt a separa train/val)
    full_dataset = JAFFEDataset(root_dir=data_dir)
    total_samples = len(full_dataset)

    if total_samples == 0:
        raise ValueError(f"Nu s-au gasit imagini valide in directorul: {data_dir}")

    # Calcularea dimensiunilor split-urilor
    train_size = int(train_ratio * total_samples)
    val_size = int(val_ratio * total_samples)
    test_size = total_samples - train_size - val_size

    # Impartirea aleatoare a indecsilor
    train_subset, val_subset, test_subset = random_split(
        full_dataset, [train_size, val_size, test_size]
    )

    # Asignarea transformarilor specifice fiecarui subset
    # Se creeaza wrapper-e usoare pentru a aplica transformari distincte pe subset-uri
    train_subset.dataset.transform = train_transforms
    val_subset.dataset.transform = val_test_transforms
    test_subset.dataset.transform = val_test_transforms

    # Crearea DataLoader-elor
    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_subset, batch_size=batch_size, shuffle=False, num_workers=2)

    print(f"Dataset JAFFE incarcat cu succes:")
    print(f" - Total imagini: {total_samples}")
    print(f" - Train split ({train_ratio*100:.0f}%): {train_size} imagini")
    print(f" - Val split ({val_ratio*100:.0f}%): {val_size} imagini")
    print(f" - Test split ({test_ratio*100:.0f}%): {test_size} imagini\n")

    return train_loader, val_loader, test_loader

In [4]:
if __name__ == "__main__":
    # Specificati calea catre folderul necomprimat al setului de date JAFFE
    DATASET_PATH = "/content/drive/MyDrive/IC2_Proiect/data"

    try:
        train_loader, val_loader, test_loader = get_jaffe_dataloaders(
            data_dir=DATASET_PATH,
            batch_size=16,
            train_ratio=0.7,
            val_ratio=0.15,
            test_ratio=0.15
        )

        # Testare functionalitate DataLoader (extragere un batch din train_loader)
        images, labels = next(iter(train_loader))

        print("=== Verificare Functionalitate DataLoader ===")
        print(f"Dimensiune Batch Imagini (B, C, H, W): {images.shape}")
        print(f"Dimensiune Batch Etichete: {labels.shape}")
        print(f"Etichetele din primul batch: {labels.tolist()}")
        print("Dataset-ul si DataLoader-ele sunt complet functionale!")

    except Exception as e:
        print(f"Eroare la verificarea dataset-ului: {e}")

Dataset JAFFE incarcat cu succes:
 - Total imagini: 213
 - Train split (70%): 149 imagini
 - Val split (15%): 31 imagini
 - Test split (15%): 33 imagini

=== Verificare Functionalitate DataLoader ===
Dimensiune Batch Imagini (B, C, H, W): torch.Size([16, 3, 224, 224])
Dimensiune Batch Etichete: torch.Size([16])
Etichetele din primul batch: [5, 4, 4, 2, 3, 5, 5, 3, 2, 1, 5, 1, 3, 0, 0, 1]
Dataset-ul si DataLoader-ele sunt complet functionale!
